[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_47_GPU_Autoscaling.ipynb)

# Lesson 47 — GPU Autoscaling: vLLM Dynamic Batching + Kubernetes HPA
### Track 5 · Production Infrastructure · Lesson 2 of 5

---

## Track 5 Roadmap

| Lesson | Topic | Status |
|--------|-------|--------|
| L46 | Durable Execution — crash-safe pipelines (SQLite → Prefect → Temporal) | ✅ Done |
| **L47** | **GPU Autoscaling — vLLM dynamic batching + Kubernetes HPA** | 📍 You are here |
| L48 | Observability — OpenTelemetry tracing for AI pipelines | ⬜ Next |
| L49 | Eval at Scale — CI evaluation harnesses for production | ⬜ |
| L50 | Track 5 Capstone — Production-hardened inference platform | ⬜ |

---

## The Core Problem: The Idle GPU Tax

In L37 you learned that vLLM gives you ~10–24× throughput over naive HuggingFace generate. But there's a harder problem: **what happens between requests?**

| GPU Type | On-demand $/hr | 24/7 monthly cost | If busy only 20% of the day |
|----------|---------------|-------------------|-----------------------------|
| T4 | \$0.35 | \$252 | \$202 wasted |
| A10G | \$0.75 | \$540 | \$432 wasted |
| A100 40GB | \$3.00 | \$2,160 | \$1,728 wasted |
| A100 80GB | \$5.00 | \$3,600 | \$2,880 wasted |

**Autoscaling is the solution.** But it only works if you understand *what to measure* and *how vLLM behaves under load.*

---

## What You'll Build Today

```
Traffic Spike                           
    │                                   
    ▼                                   
┌─────────────────────────────────────┐
│  vLLM Server (1 GPU pod)           │
│  ┌──────────────────────────────┐   │
│  │  Continuous Batching Engine  │   │
│  │  KV Cache: 85% used ⚠️      │   │
│  │  Queue depth: 47 requests   │   │
│  └──────────────────────────────┘   │
│  /metrics → Prometheus              │
└─────────────────────────────────────┘
    │                                   
    ▼ metric: vllm:num_requests_waiting > 10
┌─────────────────────────────────────┐
│  HPA / KEDA Autoscaler             │
│  current_replicas: 1                │
│  desired_replicas: 3  ⬆️           │
└─────────────────────────────────────┘
    │                                   
    ▼                                   
Pod 1 (warm) + Pod 2 (cold start 90s) + Pod 3 (cold start 90s)
```

**Key insight:** The autoscaler is simple. The hard part is choosing the right metric and handling cold-start latency.


## Part 1: Static vs Continuous Batching — Why It Matters for Autoscaling

Understanding batching is prerequisite to understanding scaling decisions.

### Static Batching (naive HuggingFace)

```
Time ──────────────────────────────────────►

Req A (10 tokens)  [████████████████████]DONE
Req B (200 tokens) [████████████████████████████████████████████]DONE
Req C (50 tokens)  [████████████████████]DONE
                    │←── Wait for ALL to finish ──────────────────►│
Next batch starts here ──────────────────────────────────────────►│
```

**Problem:** Short requests finish early but GPU sits idle waiting for the longest request in the batch.

### Continuous Batching (vLLM)

```
Time ──────────────────────────────────────►

Req A (10 tokens)  [████]DONE
Req D immediately──────►[████████]DONE
Req B (200 tokens) [████████████████████████████████████████████]DONE
Req C (50 tokens)  [████████████████]DONE
Req E immediately────────────────►[████████████]DONE
```

**The fix:** As soon as Req A finishes (10 tokens), Req D fills that GPU slot *immediately*. The GPU never waits.

### Why this affects autoscaling

With continuous batching, **GPU utilization is much higher per replica** — so you need *fewer* replicas. But when load truly spikes past what continuous batching can absorb, you see:
- `vllm:num_requests_waiting` grows (requests queued, no KV cache space)
- `vllm:gpu_cache_usage_perc` → 100% (KV cache full, can't accept new requests)
- latency increases

**These are your autoscaling signals.** Not CPU. Not memory. Not raw QPS.


In [ ]:
# ─── Setup ───────────────────────────────────────────────────────────────────
!pip install fastapi uvicorn httpx matplotlib pandas PyYAML anthropic nest_asyncio -q

import os, json, time, math, random, threading, asyncio
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable
from collections import deque

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import nest_asyncio
nest_asyncio.apply()

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

print("✅ Setup complete")

In [ ]:
# ─── Throughput Simulator: Static vs Continuous Batching ─────────────────────
#
# Model: GPU can process MAX_TOKENS_PER_SECOND tokens per second.
# Static batching: start a new batch only after ALL requests in current batch finish.
# Continuous batching: as soon as a slot frees, fill it immediately.

@dataclass
class Request:
    req_id: int
    arrival_s: float      # when request arrived
    output_tokens: int    # how many tokens to generate
    start_s: float = 0.0  # when it started generating
    finish_s: float = 0.0 # when it finished

    @property
    def latency(self):
        return self.finish_s - self.arrival_s

    @property
    def wait_time(self):
        return self.start_s - self.arrival_s


def simulate_static_batching(
    requests: List[Request],
    batch_size: int = 8,
    tokens_per_sec: float = 2000
) -> List[Request]:
    """Process requests in fixed batches. Next batch waits for slowest."""
    completed = []
    queue = sorted(requests, key=lambda r: r.arrival_s)
    clock = 0.0

    while queue:
        batch = queue[:batch_size]
        queue = queue[batch_size:]

        # Batch starts when all requests have arrived AND GPU is free
        batch_start = max(clock, max(r.arrival_s for r in batch))
        total_tokens = sum(r.output_tokens for r in batch)
        batch_duration = total_tokens / tokens_per_sec
        batch_end = batch_start + batch_duration

        for r in batch:
            r.start_s = batch_start
            r.finish_s = batch_end  # all finish at the same time (slowest dictates)
            completed.append(r)

        clock = batch_end

    return completed


def simulate_continuous_batching(
    requests: List[Request],
    max_concurrent: int = 8,
    tokens_per_sec: float = 2000
) -> List[Request]:
    """As soon as a slot frees, inject the next waiting request."""
    completed = []
    queue = sorted(requests, key=lambda r: r.arrival_s)
    in_flight: List[Request] = []
    clock = 0.0

    while queue or in_flight:
        # Fill empty slots from queue (if requests have arrived)
        while len(in_flight) < max_concurrent and queue:
            r = queue[0]
            if r.arrival_s <= clock:
                r.start_s = clock
                r.finish_s = clock + (r.output_tokens / tokens_per_sec * max_concurrent)
                # 💡 Each request gets 1/max_concurrent share of GPU bandwidth
                in_flight.append(r)
                queue.pop(0)
            else:
                break

        if not in_flight:
            # Jump clock to next arrival
            clock = queue[0].arrival_s if queue else clock
            continue

        # Advance clock to the soonest completion
        next_finish = min(r.finish_s for r in in_flight)
        if queue and queue[0].arrival_s < next_finish:
            clock = queue[0].arrival_s
        else:
            clock = next_finish
            done = [r for r in in_flight if r.finish_s <= clock]
            in_flight = [r for r in in_flight if r.finish_s > clock]
            completed.extend(done)

    return completed


# ── Generate synthetic traffic ──
random.seed(42)
N = 64  # 64 requests arriving over 30 seconds
reqs_static = [
    Request(
        req_id=i,
        arrival_s=random.uniform(0, 30),
        output_tokens=random.choice([50, 100, 200, 400, 800])  # variable length!
    ) for i in range(N)
]
# Deep copy for second simulation
reqs_continuous = [
    Request(req_id=r.req_id, arrival_s=r.arrival_s, output_tokens=r.output_tokens)
    for r in reqs_static
]

static_results = simulate_static_batching(reqs_static, batch_size=8)
cont_results   = simulate_continuous_batching(reqs_continuous, max_concurrent=8)

def stats(results, label):
    latencies = [r.latency for r in results]
    waits     = [r.wait_time for r in results]
    total_time = max(r.finish_s for r in results)
    print(f"── {label} ──")
    print(f"  Total wall time:  {total_time:.1f}s")
    print(f"  Throughput:       {N/total_time:.1f} req/s")
    print(f"  Latency p50:      {sorted(latencies)[N//2]:.1f}s")
    print(f"  Latency p95:      {sorted(latencies)[int(N*0.95)]:.1f}s")
    print(f"  Mean queue wait:  {sum(waits)/len(waits):.1f}s")
    print()

stats(static_results, "Static Batching (batch_size=8)")
stats(cont_results, "Continuous Batching (max_concurrent=8)")

# ── Visualize latency distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Latency CDFs
ax = axes[0]
for results, label, color in [
    (static_results, 'Static Batching', '#e74c3c'),
    (cont_results,   'Continuous Batching', '#27ae60')
]:
    lats = sorted(r.latency for r in results)
    ax.plot(lats, [i/N for i in range(N)], label=label, color=color, linewidth=2)
ax.axvline(x=sorted(r.latency for r in cont_results)[int(N*0.95)], color='#27ae60',
           linestyle='--', alpha=0.6, label='Continuous p95')
ax.set_xlabel('Latency (s)')
ax.set_ylabel('CDF')
ax.set_title('Latency CDF: Static vs Continuous Batching')
ax.legend()
ax.grid(True, alpha=0.3)

# Throughput by token length bucket
ax = axes[1]
buckets = {50: [], 100: [], 200: [], 400: [], 800: []}
for s, c in zip(static_results, cont_results):
    buckets[s.output_tokens].append((s.latency, c.latency))

x = list(buckets.keys())
static_means = [sum(v[0] for v in vals)/len(vals) if vals else 0 for vals in buckets.values()]
cont_means   = [sum(v[1] for v in vals)/len(vals) if vals else 0 for vals in buckets.values()]
xs = np.arange(len(x)); width = 0.35
ax.bar(xs - width/2, static_means, width, label='Static', color='#e74c3c', alpha=0.8)
ax.bar(xs + width/2, cont_means,   width, label='Continuous', color='#27ae60', alpha=0.8)
ax.set_xticks(xs)
ax.set_xticklabels([f'{t} tok' for t in x])
ax.set_ylabel('Mean Latency (s)')
ax.set_title('Mean Latency by Output Length')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 💡 EXPERIMENT: Change N=128 and see how the latency gap widens.
# 💡 EXPERIMENT: Change batch_size=16 (static) vs max_concurrent=16 (continuous).
# 💡 KEY INSIGHT: Continuous batching benefits SHORT requests most (their latency collapses).
#                 This is why autoscaling should trigger on QUEUE DEPTH not avg latency.

## Part 2: What to Measure — vLLM Prometheus Metrics

vLLM exposes a `/metrics` endpoint (Prometheus format) when started with `--enable-chunked-prefill`. Here are the metrics that matter for autoscaling:

| Metric | Type | Autoscale trigger? | Why |
|--------|------|---------------------|-----|
| `vllm:num_requests_running` | Gauge | ❌ Not directly | Just shows current in-flight count |
| `vllm:num_requests_waiting` | **Gauge** | **✅ Primary** | Queue depth = backlog = need more replicas |
| `vllm:gpu_cache_usage_perc` | **Gauge** | **✅ Secondary** | Near 100% = KV cache full = can't accept more |
| `vllm:e2e_request_latency_seconds` | Histogram | ⚠️ Lagging | Latency rises *after* the problem starts |
| `vllm:time_to_first_token_seconds` | Histogram | ⚠️ Lagging | Same issue — react too late |
| `vllm:num_requests_finished_total` | Counter | ❌ Rate only | Throughput, not capacity signal |

### Why queue depth beats latency as an autoscale signal

```
Timeline of a traffic spike:

T+0s:  Traffic doubles
T+2s:  Queue depth spikes ←── Scale-up trigger here (FAST)
T+20s: Latency p95 rises  ←── If you wait for latency, you're already 18s behind
T+90s: New pod is warm    ←── Cold start takes 90s
T+110s: Latency recovers

If you trigger on latency: users experience 90+ seconds of bad latency.
If you trigger on queue depth: only 90s of bad latency (unavoidable cold start).
```

### Scaling formula

Kubernetes HPA uses this formula:

```
desired_replicas = ceil(current_replicas × (current_metric / target_metric))
```

For vLLM, we configure:
- `target_metric = 10` (10 waiting requests per replica is our target)
- If `num_requests_waiting = 47` with 2 replicas:
  - `desired = ceil(2 × 47/10) = ceil(9.4) = 10` replicas

That's aggressive. We'll add a scale-up cap and stabilization window.


In [ ]:
# ─── Mock vLLM Metrics Server + Autoscaler Brain ─────────────────────────────
#
# We simulate a vLLM metrics endpoint and an autoscaler that polls it.
# In production: replace MockMetricsServer with real vLLM's /metrics endpoint.

from fastapi import FastAPI, Response
import uvicorn
import httpx
import threading

# ── Shared state simulating vLLM internals ──
vllm_state = {
    "num_requests_running": 0,
    "num_requests_waiting": 0,
    "gpu_cache_usage_perc": 0.0,
    "num_requests_finished_total": 0,
}

metrics_app = FastAPI()

@metrics_app.get("/metrics")
def prometheus_metrics():
    """Returns Prometheus-format metrics (simplified)."""
    lines = []
    for key, value in vllm_state.items():
        prom_name = f"vllm:{key}"
        lines.append(f"# TYPE {prom_name} gauge")
        lines.append(f"{prom_name} {value}")
    return Response(content="\n".join(lines), media_type="text/plain")

@metrics_app.get("/health")
def health():
    return {"status": "ok"}

# ── Start mock server in background ──
_server_thread = None
_server_started = threading.Event()

def _run_server():
    config = uvicorn.Config(metrics_app, host="0.0.0.0", port=8765, log_level="error")
    server = uvicorn.Server(config)
    _server_started.set()
    server.run()

_server_thread = threading.Thread(target=_run_server, daemon=True)
_server_thread.start()
time.sleep(1.5)  # wait for uvicorn to bind
print("✅ Mock vLLM metrics server running on :8765")

# ── Metrics parser ──
def parse_prometheus(text: str) -> dict:
    """Parse Prometheus text format into {metric_name: value}."""
    result = {}
    for line in text.strip().split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        parts = line.split()
        if len(parts) == 2:
            result[parts[0]] = float(parts[1])
    return result

def poll_metrics(url="http://localhost:8765/metrics") -> dict:
    resp = httpx.get(url, timeout=5)
    return parse_prometheus(resp.text)

# ── Autoscaler Brain ──
@dataclass
class ScalingDecision:
    timestamp: float
    current_replicas: int
    desired_replicas: int
    reason: str
    metrics: dict

class VLLMAutoscaler:
    def __init__(
        self,
        min_replicas: int = 1,
        max_replicas: int = 10,
        queue_depth_target: int = 10,         # scale up when waiting > N per replica
        cache_usage_scale_up: float = 0.80,   # scale up when cache > 80%
        cache_usage_scale_down: float = 0.30, # scale down when cache < 30%
        scale_up_cooldown_s: float = 90.0,    # don't scale up again for 90s (cold start)
        scale_down_cooldown_s: float = 300.0, # don't scale down for 5min
        metrics_url: str = "http://localhost:8765/metrics",
    ):
        self.min_replicas = min_replicas
        self.max_replicas = max_replicas
        self.queue_depth_target = queue_depth_target
        self.cache_usage_scale_up = cache_usage_scale_up
        self.cache_usage_scale_down = cache_usage_scale_down
        self.scale_up_cooldown_s = scale_up_cooldown_s
        self.scale_down_cooldown_s = scale_down_cooldown_s
        self.metrics_url = metrics_url
        self.current_replicas = min_replicas
        self._last_scale_up_s = 0.0
        self._last_scale_down_s = 0.0
        self.history: List[ScalingDecision] = []

    def evaluate(self, metrics: dict) -> ScalingDecision:
        now = time.time()
        waiting = metrics.get("vllm:num_requests_waiting", 0)
        cache   = metrics.get("vllm:gpu_cache_usage_perc", 0.0)
        desired = self.current_replicas
        reason = "steady"

        # ── Scale UP logic ──
        queue_ratio = waiting / (self.current_replicas * self.queue_depth_target + 1e-9)
        queue_desired = math.ceil(self.current_replicas * queue_ratio)

        scale_up_needed = (
            (waiting > self.queue_depth_target * self.current_replicas) or
            (cache > self.cache_usage_scale_up)
        )

        if scale_up_needed and (now - self._last_scale_up_s) >= self.scale_up_cooldown_s:
            desired = min(self.max_replicas, max(queue_desired, self.current_replicas + 1))
            if cache > self.cache_usage_scale_up:
                reason = f"cache_pressure ({cache:.0%} > {self.cache_usage_scale_up:.0%})"
            else:
                reason = f"queue_depth ({waiting:.0f} waiting, {queue_desired} replicas needed)"
            if desired > self.current_replicas:
                self._last_scale_up_s = now

        # ── Scale DOWN logic ──
        elif (
            waiting == 0 and
            cache < self.cache_usage_scale_down and
            self.current_replicas > self.min_replicas and
            (now - self._last_scale_down_s) >= self.scale_down_cooldown_s
        ):
            desired = max(self.min_replicas, self.current_replicas - 1)
            reason = f"low_load (cache={cache:.0%}, queue=0)"
            if desired < self.current_replicas:
                self._last_scale_down_s = now

        decision = ScalingDecision(
            timestamp=now,
            current_replicas=self.current_replicas,
            desired_replicas=desired,
            reason=reason,
            metrics={"waiting": waiting, "cache_pct": cache},
        )
        self.current_replicas = desired
        self.history.append(decision)
        return decision

    def poll_and_decide(self) -> ScalingDecision:
        metrics = poll_metrics(self.metrics_url)
        return self.evaluate(metrics)


# ── Simulate a traffic scenario ──
print("Simulating traffic spike scenario...")
print()

scaler = VLLMAutoscaler(min_replicas=1, max_replicas=8,
                        scale_up_cooldown_s=0, scale_down_cooldown_s=0)  # disable cooldowns for demo

traffic_scenarios = [
    ("Normal load",   {"vllm:num_requests_waiting": 2,  "vllm:gpu_cache_usage_perc": 0.45}),
    ("Load rising",   {"vllm:num_requests_waiting": 8,  "vllm:gpu_cache_usage_perc": 0.60}),
    ("Spike!",        {"vllm:num_requests_waiting": 47, "vllm:gpu_cache_usage_perc": 0.85}),
    ("Still high",    {"vllm:num_requests_waiting": 35, "vllm:gpu_cache_usage_perc": 0.72}),
    ("Recovering",    {"vllm:num_requests_waiting": 12, "vllm:gpu_cache_usage_perc": 0.50}),
    ("Back to normal",{"vllm:num_requests_waiting": 0,  "vllm:gpu_cache_usage_perc": 0.20}),
    ("Quiet",         {"vllm:num_requests_waiting": 0,  "vllm:gpu_cache_usage_perc": 0.10}),
]

print(f"{'Scenario':<20} {'Wait Queue':>10} {'Cache':>8} {'Before':>8} {'After':>8}  Reason")
print("-" * 80)
for name, metrics in traffic_scenarios:
    before = scaler.current_replicas
    d = scaler.evaluate(metrics)
    change = "⬆️ " if d.desired_replicas > before else ("⬇️ " if d.desired_replicas < before else "──")
    print(f"{name:<20} {metrics['vllm:num_requests_waiting']:>10.0f} "
          f"{metrics['vllm:gpu_cache_usage_perc']:>7.0%} "
          f"{before:>7}  {change}{d.desired_replicas:<5}  {d.reason}")

# 💡 EXPERIMENT: Set scale_up_cooldown_s=90 and re-run — see how cooldown prevents thrashing.
# 💡 EXPERIMENT: Change queue_depth_target=5 (more aggressive) vs 20 (more conservative).

## Part 3: Kubernetes HPA Configuration

### Architecture: How metrics flow to HPA

```
vLLM Pod                    Prometheus              K8s API Server
────────                    ──────────              ──────────────
  /metrics ──────────────►  scrape every 15s  ──►  Prometheus Adapter
  (Prom format)                                     (custom.metrics.k8s.io)
                                                         │
                                                         ▼
                                                    HPA Controller
                                                    polls every 15s
                                                         │
                                                         ▼
                                                    Scale Deployment
                                                    (add/remove pods)
```

### Two HPA variants

**Option A — HPA v2 with Prometheus Adapter** (standard but complex setup):
- Requires installing `prometheus-adapter` in cluster
- Exposes custom metrics to `custom.metrics.k8s.io` API
- HPA reads `vllm_num_requests_waiting` as a custom metric

**Option B — KEDA** (recommended for AI workloads):
- Single operator that handles everything including **scale-to-zero**
- Built-in Prometheus trigger — no custom adapter needed
- Supports `minReplicaCount: 0` (pay nothing when idle)

We'll generate YAML for both.


In [ ]:
# ─── Kubernetes HPA + KEDA YAML Generator ────────────────────────────────────
import yaml

def generate_vllm_deployment(model_name: str, gpu_type: str = "nvidia.com/gpu") -> dict:
    """Generate a vLLM Deployment spec."""
    return {
        "apiVersion": "apps/v1",
        "kind": "Deployment",
        "metadata": {
            "name": "vllm-server",
            "labels": {"app": "vllm-server"}
        },
        "spec": {
            "replicas": 1,
            "selector": {"matchLabels": {"app": "vllm-server"}},
            "template": {
                "metadata": {"labels": {"app": "vllm-server"}},
                "spec": {
                    "containers": [{
                        "name": "vllm",
                        "image": "vllm/vllm-openai:latest",
                        "args": [
                            f"--model={model_name}",
                            "--dtype=bfloat16",
                            "--max-model-len=4096",
                            "--gpu-memory-utilization=0.90",
                            "--enable-chunked-prefill",
                            "--served-model-name=llm"
                        ],
                        "ports": [{"containerPort": 8000}],
                        "resources": {
                            "limits": {gpu_type: 1},
                            "requests": {gpu_type: 1}
                        },
                        "readinessProbe": {
                            "httpGet": {"path": "/health", "port": 8000},
                            "initialDelaySeconds": 90,  # cold start time!
                            "periodSeconds": 10,
                        },
                        "env": [
                            {"name": "HUGGING_FACE_HUB_TOKEN",
                             "valueFrom": {"secretKeyRef": {"name": "hf-token", "key": "token"}}}
                        ]
                    }]
                }
            }
        }
    }


def generate_hpa_v2(queue_depth_target: int = 10, max_replicas: int = 8) -> dict:
    """HPA v2 using Prometheus Adapter custom metric."""
    return {
        "apiVersion": "autoscaling/v2",
        "kind": "HorizontalPodAutoscaler",
        "metadata": {"name": "vllm-hpa"},
        "spec": {
            "scaleTargetRef": {
                "apiVersion": "apps/v1",
                "kind": "Deployment",
                "name": "vllm-server"
            },
            "minReplicas": 1,
            "maxReplicas": max_replicas,
            "metrics": [
                {
                    # Primary: queue depth (leading indicator)
                    "type": "External",
                    "external": {
                        "metric": {
                            "name": "vllm_num_requests_waiting",
                            "selector": {"matchLabels": {"app": "vllm-server"}}
                        },
                        "target": {
                            "type": "AverageValue",
                            "averageValue": str(queue_depth_target)
                        }
                    }
                },
                {
                    # Secondary: KV cache pressure
                    "type": "External",
                    "external": {
                        "metric": {
                            "name": "vllm_gpu_cache_usage_perc",
                            "selector": {"matchLabels": {"app": "vllm-server"}}
                        },
                        "target": {
                            "type": "AverageValue",
                            "averageValue": "0.80"  # scale when > 80% KV cache used
                        }
                    }
                }
            ],
            "behavior": {
                "scaleUp": {
                    "stabilizationWindowSeconds": 0,  # scale up immediately
                    "policies": [
                        {"type": "Pods", "value": 3, "periodSeconds": 60},  # max +3 pods/min
                        {"type": "Percent", "value": 100, "periodSeconds": 60}
                    ],
                    "selectPolicy": "Max"
                },
                "scaleDown": {
                    "stabilizationWindowSeconds": 300,  # wait 5min before scale-down
                    "policies": [
                        {"type": "Pods", "value": 1, "periodSeconds": 120}  # max -1 pod/2min
                    ]
                }
            }
        }
    }


def generate_keda_scaledobject(
    prometheus_url: str,
    queue_threshold: int = 10,
    min_replicas: int = 0,  # 0 = scale-to-zero!
    max_replicas: int = 8,
    cooldown_period_s: int = 300
) -> dict:
    """KEDA ScaledObject — recommended for LLM workloads (supports scale-to-zero)."""
    return {
        "apiVersion": "keda.sh/v1alpha1",
        "kind": "ScaledObject",
        "metadata": {"name": "vllm-keda-scaler"},
        "spec": {
            "scaleTargetRef": {
                "apiVersion": "apps/v1",
                "kind": "Deployment",
                "name": "vllm-server"
            },
            "minReplicaCount": min_replicas,  # 0 = scale-to-zero when idle
            "maxReplicaCount": max_replicas,
            "cooldownPeriod": cooldown_period_s,
            "pollingInterval": 15,  # check metrics every 15 seconds
            "triggers": [
                {
                    "type": "prometheus",
                    "metadata": {
                        "serverAddress": prometheus_url,
                        "metricName": "vllm_num_requests_waiting",
                        "threshold": str(queue_threshold),
                        "query": 'sum(vllm:num_requests_waiting{app="vllm-server"})'
                    }
                },
                {
                    "type": "prometheus",
                    "metadata": {
                        "serverAddress": prometheus_url,
                        "metricName": "vllm_gpu_cache_pressure",
                        "threshold": "0.80",
                        "query": 'max(vllm:gpu_cache_usage_perc{app="vllm-server"})'
                    }
                }
            ],
            "advanced": {
                "horizontalPodAutoscalerConfig": {
                    "behavior": {
                        "scaleUp": {
                            "stabilizationWindowSeconds": 0,
                            "policies": [{"type": "Pods", "value": 3, "periodSeconds": 60}]
                        },
                        "scaleDown": {
                            "stabilizationWindowSeconds": cooldown_period_s,
                            "policies": [{"type": "Pods", "value": 1, "periodSeconds": 120}]
                        }
                    }
                }
            }
        }
    }


# ── Print generated configs ──
print("=" * 60)
print("1. vLLM Deployment (Qwen2.5-7B-Instruct)")
print("=" * 60)
deployment = generate_vllm_deployment("Qwen/Qwen2.5-7B-Instruct-AWQ")
print(yaml.dump(deployment, default_flow_style=False, sort_keys=False))

print("=" * 60)
print("2. KEDA ScaledObject (recommended — scale-to-zero capable)")
print("=" * 60)
keda = generate_keda_scaledobject(
    prometheus_url="http://prometheus-operated.monitoring:9090",
    queue_threshold=10,
    min_replicas=0,   # Scale to ZERO when idle!
    max_replicas=8,
    cooldown_period_s=300,
)
print(yaml.dump(keda, default_flow_style=False, sort_keys=False))

print("# Deploy with:")
print("# kubectl apply -f vllm-deployment.yaml")
print("# kubectl apply -f keda-scaledobject.yaml")
print("# kubectl get hpa  # watch scaling events")

## Part 4: Scale-to-Zero — Pay Only for What You Use

With `minReplicaCount: 0`, your vLLM deployment scales down to **zero pods** when there are no requests. You pay nothing.

### The trade-off: cold start latency

| Stage | Time | What's happening |
|-------|------|------------------|
| Kubernetes pod scheduling | 5–15s | K8s finds a node with a free GPU |
| Docker image pull | 0–120s | If image not cached on node (first time) |
| CUDA driver init | 5–10s | PyTorch initializes CUDA context |
| Model weights download | 30–600s | Depends on model size and HuggingFace speed |
| KV cache pre-allocation | 5–15s | vLLM allocates GPU memory |
| **Total cold start** | **~90s** | For a 7B model with pre-pulled image |

### Strategies to handle cold start

```
Strategy 1: Accept cold starts (best for batch/async workloads)
   minReplicas=0, no mitigation
   ✅ Maximum savings  ❌ Bad for interactive users

Strategy 2: Keep 1 warm replica (best for interactive workloads)
   minReplicas=1, scale additional replicas on demand
   ✅ No cold starts for users  ❌ Pays for 1 GPU even when idle

Strategy 3: Predictive pre-warming (best for predictable traffic)
   Schedule a CronJob to scale up before known traffic peaks
   kubectl scale deployment/vllm-server --replicas=3  (runs at 8:45am)
   ✅ No cold starts during peak  ❌ Requires traffic knowledge

Strategy 4: Request queuing with async response
   Add a message queue (Redis/RabbitMQ) in front of vLLM
   Users get a job_id, poll for results
   ✅ Users don't time out during cold start  ❌ Changes API contract
```

### When to use scale-to-zero

| Workload type | Scale-to-zero? | Min replicas recommendation |
|---------------|----------------|-----------------------------|
| Batch processing / nightly jobs | ✅ Yes | 0 |
| Internal tools (dev/staging) | ✅ Yes | 0 |
| Interactive chat (non-SLA) | ⚠️ Maybe | 0 with request queuing |
| Interactive chat (SLA <2s) | ❌ No | 1 (pay for warm standby) |
| Production inference API | ❌ No | 1–2 depending on traffic |


In [ ]:
# ─── Cost Model: Static Allocation vs Autoscaling ────────────────────────────

@dataclass
class GPUCostModel:
    # On-demand spot prices ($/hr) — approximate, varies by cloud + region
    HOURLY_PRICES = {
        "T4":       {"on_demand": 0.35, "spot": 0.12},
        "A10G":     {"on_demand": 0.75, "spot": 0.30},
        "A100_40G": {"on_demand": 3.00, "spot": 1.20},
        "A100_80G": {"on_demand": 5.00, "spot": 2.00},
        "H100":     {"on_demand": 12.0, "spot": 5.00},
    }
    # Cold start time in seconds (model weight download cached on node)
    COLD_START_S = {
        "0.5B":  20,
        "1.5B":  35,
        "7B":    90,
        "13B":  150,
        "70B":  480,
    }

    gpu_type: str = "A10G"
    model_size: str = "7B"
    use_spot: bool = False

    @property
    def hourly_rate(self) -> float:
        tier = "spot" if self.use_spot else "on_demand"
        return self.HOURLY_PRICES[self.gpu_type][tier]

    @property
    def cold_start_s(self) -> float:
        return self.COLD_START_S.get(self.model_size, 90)

    def static_monthly_cost(self, num_replicas: int) -> float:
        """Always-on: pay 24/7 for N replicas."""
        return self.hourly_rate * 24 * 30 * num_replicas

    def autoscale_monthly_cost(
        self,
        min_replicas: int,
        peak_replicas: int,
        peak_hours_per_day: float,  # hours/day at peak
        off_peak_hours_per_day: float = None,  # if None, 24 - peak_hours
    ) -> dict:
        """Autoscaling: pay for min_replicas off-peak + peak_replicas at peak."""
        if off_peak_hours_per_day is None:
            off_peak_hours_per_day = 24 - peak_hours_per_day
        ramp_hours = (self.cold_start_s / 3600) * 30  # cold start cost per month
        peak_cost = self.hourly_rate * peak_hours_per_day * 30 * peak_replicas
        off_peak_cost = self.hourly_rate * off_peak_hours_per_day * 30 * min_replicas
        total = peak_cost + off_peak_cost
        return {
            "total": total,
            "peak_cost": peak_cost,
            "off_peak_cost": off_peak_cost,
            "cold_start_hrs_per_month": ramp_hours,
        }

    def compare(
        self,
        static_replicas: int,
        min_replicas: int,
        peak_replicas: int,
        peak_hours_per_day: float
    ):
        static = self.static_monthly_cost(static_replicas)
        auto = self.autoscale_monthly_cost(min_replicas, peak_replicas, peak_hours_per_day)
        savings = static - auto["total"]
        savings_pct = savings / static * 100 if static > 0 else 0
        print(f"  GPU:          {self.gpu_type} @ \${self.hourly_rate:.2f}/hr ({'spot' if self.use_spot else 'on-demand'})")
        print(f"  Model:        {self.model_size} (cold start ~{self.cold_start_s}s)")
        print(f"  Peak traffic: {peak_hours_per_day}h/day → {peak_replicas} replicas")
        print(f"  Off-peak:     {24-peak_hours_per_day}h/day → {min_replicas} replica(s)")
        print()
        print(f"  Static ({static_replicas} always-on):   \${static:,.0f}/month")
        print(f"  Autoscale:              \${auto['total']:,.0f}/month")
        print(f"  ├─ Peak cost:           \${auto['peak_cost']:,.0f}")
        print(f"  └─ Off-peak cost:       \${auto['off_peak_cost']:,.0f}")
        print(f"  Monthly savings:        \${savings:,.0f} ({savings_pct:.1f}%)")
        return {"static": static, "autoscale": auto["total"], "savings": savings, "savings_pct": savings_pct}


# ── Run comparisons across different scenarios ──
scenarios = [
    {
        "name": "Internal tool (dev team)",
        "gpu": "T4", "model": "7B", "spot": False,
        "static_replicas": 2, "min": 0, "peak": 2, "peak_hours": 10
    },
    {
        "name": "Production chat API (business hours)",
        "gpu": "A10G", "model": "7B", "spot": False,
        "static_replicas": 4, "min": 1, "peak": 4, "peak_hours": 14
    },
    {
        "name": "Large model inference (A100)",
        "gpu": "A100_40G", "model": "70B", "spot": False,
        "static_replicas": 2, "min": 0, "peak": 2, "peak_hours": 8
    },
    {
        "name": "Batch processing (overnight)",
        "gpu": "A100_80G", "model": "70B", "spot": True,
        "static_replicas": 4, "min": 0, "peak": 4, "peak_hours": 6
    },
]

results = []
for s in scenarios:
    print(f"\n{'='*55}")
    print(f"Scenario: {s['name']}")
    print('='*55)
    model = GPUCostModel(gpu_type=s['gpu'], model_size=s['model'], use_spot=s['spot'])
    r = model.compare(
        static_replicas=s['static_replicas'],
        min_replicas=s['min'],
        peak_replicas=s['peak'],
        peak_hours_per_day=s['peak_hours']
    )
    results.append({"name": s['name'], **r})

# ── Summary bar chart ──
fig, ax = plt.subplots(figsize=(12, 5))
names = [r['name'].split('(')[0].strip() for r in results]
x = np.arange(len(names)); width = 0.35
bars1 = ax.bar(x - width/2, [r['static'] for r in results], width, label='Static (always-on)', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x + width/2, [r['autoscale'] for r in results], width, label='Autoscale', color='#27ae60', alpha=0.85)
for bar, r in zip(bars2, results):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f"-{r['savings_pct']:.0f}%", ha='center', va='bottom', fontsize=9, color='#27ae60', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Monthly GPU Cost (USD)')
ax.set_title('Static vs Autoscale GPU Cost Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# 💡 EXPERIMENT: Change peak_hours to 20 (high-traffic business) and see savings shrink.
# 💡 EXPERIMENT: Set use_spot=True — spot instances cut the savings calculation further.
# 💡 KEY INSIGHT: Autoscaling saves most when traffic is BURSTY, not steady-state.

## Part 5: Pitfalls — 10 Ways Autoscaling Goes Wrong

| # | Pitfall | Symptom | Fix |
|---|---------|---------|-----|
| 1 | **Cold start kills SLA** | P99 latency spikes to 90s+ on scale-up | Keep `minReplicas=1` for interactive; use request queuing for async |
| 2 | **Scaling on CPU/memory** | HPA fires constantly, pods scaled but GPU still bottleneck | Scale ONLY on `vllm:num_requests_waiting` and `gpu_cache_usage_perc` |
| 3 | **No scale-up cooldown** | Thrashing — scale to 8 pods, immediately scale back to 1 | Set `stabilizationWindowSeconds: 60` for scale-up; 300 for scale-down |
| 4 | **readinessProbe too short** | Pod gets traffic before model loaded → timeout errors | Set `initialDelaySeconds` ≥ cold start time (90s for 7B) |
| 5 | **Scaling on avg latency** | React 30s after the problem started, latency already bad | Use queue depth (leading indicator), not latency (lagging) |
| 6 | **Scale-to-zero + synchronous API** | First user after idle gets 90s timeout | Either `minReplicas=1` OR switch to async queue + polling |
| 7 | **Single Prometheus query across all models** | Wrong scale signal when multiple models share a cluster | Filter metrics by `app` and `model_name` labels |
| 8 | **Spot instance eviction mid-request** | Request in-flight when GPU node is reclaimed | Set `gpu-memory-utilization=0.85` (leave headroom), use PodDisruptionBudget |
| 9 | **HPA fights KEDA** | Both trying to control `spec.replicas` → random oscillation | Use ONLY ONE autoscaler. KEDA subsumes HPA — don't install both |
| 10 | **No max_replicas cap** | Runaway scaling event → $10K cloud bill overnight | Always set `maxReplicaCount`, AND set billing alerts in cloud console |

### Pitfall #1 is the most important to design for upfront

The decision between `minReplicas=0` vs `minReplicas=1` is an **architecture decision**, not a config tweak. Once you build your client to expect synchronous responses, scale-to-zero requires changing the API contract.


In [ ]:
# ─── Pitfall Demo: Cold Start Latency vs Warm Pool ───────────────────────────
#
# Simulate: What happens to P99 latency when pods scale from zero?

import random
from typing import Tuple

@dataclass
class PodPool:
    warm_pods: int = 1           # pods already running
    cold_start_s: float = 90.0   # seconds for new pod to become ready
    base_latency_s: float = 0.5  # latency when pod is warm
    max_concurrent_per_pod: int = 8

    def __post_init__(self):
        self._in_flight = [0] * max(self.warm_pods, 1)  # requests per pod
        self._pod_ready_at = [0.0] * max(self.warm_pods, 1)  # 0 = already warm

    def handle_request(self, clock: float) -> Tuple[float, str]:
        """Returns (latency_s, pod_status)."""
        # Find pod with free slot
        for i, (inflight, ready_at) in enumerate(zip(self._in_flight, self._pod_ready_at)):
            if ready_at <= clock and inflight < self.max_concurrent_per_pod:
                self._in_flight[i] += 1
                latency = self.base_latency_s + random.gauss(0, 0.1)
                self._in_flight[i] -= 1
                return max(0.1, latency), "warm"

        # No warm pod available — new pod spinning up
        new_ready_at = clock + self.cold_start_s
        self._pod_ready_at.append(new_ready_at)
        self._in_flight.append(1)
        latency = self.cold_start_s + self.base_latency_s
        self._in_flight[-1] -= 1
        return latency, "cold_start"


def simulate_traffic(pool: PodPool, req_per_second: float, duration_s: float):
    """Simulate traffic arriving at req_per_second for duration_s seconds."""
    latencies = []
    statuses = []
    clock = 0.0
    while clock < duration_s:
        latency, status = pool.handle_request(clock)
        latencies.append(latency)
        statuses.append(status)
        # Arrival interval with some jitter (Poisson-like)
        clock += random.expovariate(req_per_second)
    return latencies, statuses


# ── Scenario 1: Scale-to-zero (0 warm pods, cold start on first request) ──
pool_zero = PodPool(warm_pods=0, cold_start_s=90.0, base_latency_s=0.5)
lats_zero, stats_zero = simulate_traffic(pool_zero, req_per_second=5, duration_s=120)

# ── Scenario 2: 1 warm pod (min_replicas=1) ──
pool_warm = PodPool(warm_pods=1, cold_start_s=90.0, base_latency_s=0.5)
lats_warm, stats_warm = simulate_traffic(pool_warm, req_per_second=5, duration_s=120)

# ── Scenario 3: Request queue (async — users get job_id, cold start is hidden) ──
# Simulate: ALL requests get base latency (queue absorbs cold start)
pool_queue = PodPool(warm_pods=0, cold_start_s=0.0, base_latency_s=0.5)  # cold start hidden
lats_queue, _ = simulate_traffic(pool_queue, req_per_second=5, duration_s=120)

def percentile(arr, p):
    return sorted(arr)[int(len(arr) * p / 100)]

print("Cold Start Impact on Latency")
print(f"{'Strategy':<30} {'p50':>8} {'p95':>8} {'p99':>8} {'Cold starts':>12}")
print("-" * 70)
for name, lats, stats_list in [
    ("Scale-to-zero (0 warm)", lats_zero, stats_zero),
    ("1 warm pod (min_replicas=1)", lats_warm, stats_warm),
    ("Async queue (cold hidden)", lats_queue, []),
]:
    cold_count = sum(1 for s in stats_list if s == "cold_start")
    print(f"{name:<30} {percentile(lats,50):>7.1f}s {percentile(lats,95):>7.1f}s "
          f"{percentile(lats,99):>7.1f}s {cold_count:>12}")

# ── Latency distribution plot ──
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
scenario_data = [
    ("Scale-to-zero\n(minReplicas=0)", lats_zero, "#e74c3c"),
    ("1 warm pod\n(minReplicas=1)", lats_warm, "#f39c12"),
    ("Async queue\n(cold start hidden)", lats_queue, "#27ae60"),
]
for ax, (title, lats, color) in zip(axes, scenario_data):
    display_lats = [min(l, 15) for l in lats]  # cap at 15s for visualization
    ax.hist(display_lats, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(percentile(lats, 95), color='black', linestyle='--', linewidth=1.5, label=f'p95={percentile(lats,95):.1f}s')
    ax.set_title(title)
    ax.set_xlabel('Latency (s, capped at 15s)')
    ax.set_ylabel('Request count')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Latency Distribution by Scale-to-Zero Strategy', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print()
print("Key takeaway: Scale-to-zero destroys P99 for SYNCHRONOUS APIs.")
print("Use min_replicas=1 OR switch to async if you need scale-to-zero savings.")

## Summary: What You Built

| Component | What it does | Key design choice |
|-----------|--------------|-------------------|
| Throughput simulator | Shows why continuous batching ≫ static batching | Benefits SHORT requests most |
| Autoscaler brain | Reads vLLM metrics → scaling decisions | Scale on `num_requests_waiting`, not latency |
| HPA YAML (K8s v2) | Standard K8s autoscaling | Needs Prometheus Adapter installed |
| KEDA ScaledObject | Modern autoscaling with scale-to-zero | Recommended — no adapter needed |
| Cost model | Static vs autoscale comparison | Savings largest for bursty traffic |
| Cold start demo | Why P99 explodes with scale-to-zero | Use `minReplicas=1` or async queue |

---

## 5 Homework Tasks

1. **Throughput curve**: Modify `simulate_continuous_batching` to plot throughput (req/s) vs `max_concurrent` from 1 to 32. Find the elbow point where adding concurrency stops helping.

2. **Hysteresis**: Add a hysteresis band to `VLLMAutoscaler` — only scale down when queue has been 0 for 5 consecutive polls. Compare scaling oscillation vs. the naive version.

3. **Multi-model cluster**: Extend the cost model to support 3 models (large/medium/small) sharing a GPU node pool. Show how model routing (send simple queries to small) reduces GPU count.

4. **Prometheus Adapter config**: Research the `prometheus-adapter` ConfigMap format. Write the `seriesQuery` and `metricsQuery` rules that expose `vllm:num_requests_waiting` as a K8s custom metric.

5. **CronJob pre-warmer**: Write the YAML for a Kubernetes CronJob that runs `kubectl scale deployment/vllm-server --replicas=4` at 8:45am every weekday. What RBAC role does it need?

---

## L48 Preview: OpenTelemetry Tracing for AI Pipelines

In L47 you can see *that* scaling is happening. In L48 you'll see *why* — distributed traces that follow a single request from API gateway → orchestrator → vLLM pod → tool call → response. We'll wire OpenTelemetry spans into the durable pipeline from L46 and the multi-agent swarm from L36.

```
Trace: research_query (total: 4.2s)
├── orchestrator.plan      (120ms)
├── a2a.searcher_agent     (1.8s)
│   ├── vllm.generate      (1.5s)  ← your bottleneck is here
│   └── tool.web_search    (300ms)
├── a2a.critic_agent       (900ms)
└── durable.persist_step   (12ms)
```

Without traces, you're guessing. With traces, the bottleneck is obvious.
